# Ctypes And Memory-Mapped Files

## Quick notes

### What this covers
Two ways to work on raw memory: `ctypes` (call functions in compiled C libraries from Python, no compiler needed) and `mmap` (treat a file or anonymous memory as a mutable byte block). Plus `struct`, the buffer protocol and `memoryview` as the glue between them.

### Key points
- **Loading**: `ctypes.util.find_library("c")` returns a name or `None`; `ctypes.CDLL(name)` loads it; `CDLL(None)` is "this process" on Unix. Always guard both steps.
- **Types**: `c_int`, `c_double`, `c_char_p`, `c_void_p`, `c_size_t`... Read/write with `.value`. They wrap silently on overflow (`c_ubyte(256).value == 0`).
- **Always set `fn.argtypes` and `fn.restype`.** Defaults assume int in and int out. With `argtypes`, bad calls raise `ArgumentError` instead of corrupting memory.
- **Pointers**: `byref(x)` for a temporary address in a call, `pointer(x)` for an object you keep (`.contents`, `p[0]`), `POINTER(c_int)` is the type.
- **Arrays**: `(c_int * 5)(...)`: fixed size, indexable, sliceable. `from_buffer(bytearray)` shares memory.
- **`Structure`** with `_fields_` mirrors a C struct including padding; `_pack_ = 1` removes padding; `Union` overlays fields.
- **`create_string_buffer(n)`** is the mutable buffer C can write into (`.value` stops at NUL, `.raw` is everything).
- **Callbacks**: `CFUNCTYPE(restype, *argtypes)(py_func)`; `qsort` calls your compare function. Keep the wrapper alive.
- **FFI choices**: ctypes for a few functions; cffi for bigger APIs; Cython for fast loops; a C extension or pybind11/PyO3 for full control.
- **`mmap.mmap(-1, n)`** anonymous; `mmap.mmap(f.fileno(), 0)` file-backed whole file. Index gives an int, slice gives a bytes copy, `find`/`rfind`, `read`/`write`/`seek`, `re.search` works on it.
- **Access modes**: `ACCESS_READ`, `ACCESS_WRITE`, `ACCESS_COPY` (private copy-on-write).
- **`flush()`** pushes changes to disk; `resize()` changes mapping and file (not supported everywhere); `madvise()` is a Unix-only performance hint.
- **`struct`**: `pack/unpack/calcsize`, `<` `>` `=` `@`, `pack_into/unpack_from` straight on an mmap; `struct.Struct` precompiles a format.
- **Buffer protocol**: bytes, bytearray, mmap, `array`, ctypes arrays share memory with no copy; `memoryview` slices are zero-copy. See `../02-Data Types/10-Bytes-And-Bytearray.ipynb`.

### Gotchas (the part worth revising twice)
- **Segfaults give no traceback.** Wrong `argtypes`, wrong struct layout, NULL pointers or a dead callback kill the interpreter (exit code 139). Test risky calls in a throwaway subprocess.
- **Variadic C functions** (`printf`, `sscanf`, `snprintf`) can crash through plain ctypes on some platforms (Apple Silicon). Use fixed-argument functions.
- **Default `restype` is a 32-bit int**, which truncates 64-bit pointers. Set `restype = c_void_p`.
- **Keep callbacks referenced** while C might call them; exceptions inside a callback are printed and swallowed, never raised to the caller.
- **`c_char_p` from Python bytes points into the immutable bytes**; never write through it. Use `create_string_buffer`.
- **Live views block `close()`**: a `from_buffer` ctypes object or memoryview on an mmap makes `close()` raise `BufferError`. `del` the view (or `.release()`) first.
- **Empty file** cannot be mapped (`ValueError`); mapping offsets must be multiples of `ALLOCATIONGRANULARITY`; use after `close()` raises `ValueError`.
- **Another process shrinking the file** under your mapping can crash you (SIGBUS). Two writers need their own locking.
- **Slices copy** (`m[a:b]` is bytes); use `memoryview(m)` for zero-copy.
- **Native `struct` format (`@`) pads**; use `<`/`>`/`=` for file and network layouts.
- **Library names and C type sizes differ by OS**, so never hard-code `libc.so.6`.

### When it matters
Calling a vendor or OS library that has no Python wrapper, parsing or patching big binary files, sharing a buffer between processes, reading file headers as structs. For everything else, pure Python or an existing wrapper is safer.

### One-line recap example
```python
import ctypes, ctypes.util
libm = ctypes.CDLL(ctypes.util.find_library("m")); libm.sqrt.argtypes = [ctypes.c_double]; libm.sqrt.restype = ctypes.c_double
print(libm.sqrt(16.0))   # 4.0
```

In [1]:
import ctypes
import ctypes.util
import mmap
import os
import struct
import sys
import tempfile

## 01. Index - Talking To C And To Raw Memory

- 01 Index                      12 mmap: anonymous
- 02 Why ctypes (and the rivals) 13 mmap: file-backed
- 03 Loading a C library        14 mmap: slicing and find
- 04 Fundamental C types        15 mmap: resize and flush
- 05 byref, pointer and POINTER 16 mmap: madvise (guarded)
- 06 Setting argtypes/restype   17 struct recap for binary layouts
- 07 Arrays                     18 Buffer protocol and memoryview
- 08 Structure                  19 ctypes + mmap together
- 09 create_string_buffer       20 Safety gotchas
- 10 Callbacks: qsort           21 Decision guide
- 11 Errors and sizes
- Two ideas in one file: ctypes lets Python call functions inside compiled
  C libraries; mmap lets Python treat a file (or anonymous memory) as bytes.
  Both work on RAW MEMORY, so both can crash Python if misused.

## 02. Why ctypes (and the rivals)

- ctypes = a "foreign function interface" (FFI) in the standard library.
  You load a shared library at runtime and call functions with no C compiler.
- Good for: quick calls into an existing .so/.dylib/.dll, poking at OS APIs,
  prototyping, tests.
- Not good for: big APIs (you must declare every type by hand), speed in a
  hot loop (each call has overhead), anything that needs compile-time checks.
- Rivals (not run here, they are third-party or need a compiler):
    cffi        -> reads C declarations, often cleaner than ctypes
    Cython      -> write Python-like code that compiles to C, fast loops
    C extension -> hand-written code using the Python C API, most control
    pybind11 / PyO3 / nanobind -> C++ / Rust bindings
- Rule of thumb: ctypes for a handful of functions, cffi/Cython/extension when
  the binding grows or speed matters.

In [2]:
print("ctypes is stdlib FFI; no compiler needed:", hasattr(ctypes, "CDLL"))
print("pointer size in bits:", ctypes.sizeof(ctypes.c_void_p) * 8 in (32, 64))

ctypes is stdlib FFI; no compiler needed: True
pointer size in bits: True


## 03. Loading a C library

- ctypes.util.find_library("c") searches for a library by short name and
  returns a name/path string, or None if it cannot find one.
- ctypes.CDLL(name) loads it. ctypes.CDLL(None) means "the running program
  itself" on Unix, which already includes libc.
- Library names differ by OS (libc.so.6, libc.dylib, msvcrt.dll). Never
  hard-code them; and always guard, because the lookup can return None.

In [3]:
libc = None
libm = None
try:
    libc_name = ctypes.util.find_library("c")
    libc = ctypes.CDLL(libc_name) if libc_name else (ctypes.CDLL(None) if os.name == "posix" else None)
except OSError:
    libc = None
try:
    libm_name = ctypes.util.find_library("m")
    libm = ctypes.CDLL(libm_name) if libm_name else (ctypes.CDLL(None) if os.name == "posix" else None)
except OSError:
    libm = None
print("libc loaded:", libc is not None)
print("libm loaded:", libm is not None)
print("find_library for a nonsense name:", ctypes.util.find_library("no_such_lib_xyz"))

libc loaded: True
libm loaded: True
find_library for a nonsense name: None


## 04. Fundamental C types

- ctypes has a class for each C type: c_int, c_uint, c_long, c_longlong,
  c_float, c_double, c_char, c_char_p (bytes pointer), c_wchar_p, c_bool,
  c_void_p, c_size_t, c_ubyte ...
- Each instance holds a value; read/write it with .value.
- They do NOT behave like Python ints: no overflow errors, they wrap silently.

In [4]:
i = ctypes.c_int(41)
i.value += 1
print("c_int value:", i.value)
print("sizes (int, long long, double):", ctypes.sizeof(ctypes.c_int), ctypes.sizeof(ctypes.c_longlong), ctypes.sizeof(ctypes.c_double))
print("c_ubyte wraps 256 ->", ctypes.c_ubyte(256).value)
print("c_int wraps 2**31 ->", ctypes.c_int(2**31).value)
d = ctypes.c_double(2.5)
print("c_double:", d.value)
print("c_char_p holds bytes:", ctypes.c_char_p(b"hi").value)

c_int value: 42
sizes (int, long long, double): 4 8 8
c_ubyte wraps 256 -> 0
c_int wraps 2**31 -> -2147483648
c_double: 2.5
c_char_p holds bytes: b'hi'


## 05. byref, pointer and POINTER

- C functions often want a pointer so they can write a result back.
- byref(x)   = cheap, temporary "address of x", good for passing to a call.
- pointer(x) = a real pointer object you keep; p.contents is the target.
- POINTER(c_int) is the TYPE "pointer to int"; pointer(x) makes an instance.
- p[0] reads through the pointer, like C's *p.

In [5]:
n = ctypes.c_int(10)
p = ctypes.pointer(n)
p.contents.value = 99
print("changed through pointer:", n.value)
print("p[0]:", p[0])
print("type name:", ctypes.POINTER(ctypes.c_int).__name__)
if libm:
    # frexp(x, &exp) returns a mantissa and writes the exponent through a pointer
    libm.frexp.argtypes = [ctypes.c_double, ctypes.POINTER(ctypes.c_int)]
    libm.frexp.restype = ctypes.c_double
    exp = ctypes.c_int(0)
    mant = libm.frexp(8.0, ctypes.byref(exp))
    print("frexp(8.0) mantissa/exponent:", mant, exp.value)
# - Do not call VARIADIC C functions (printf, sscanf, snprintf) through plain
#   ctypes: on some platforms (Apple Silicon) their calling convention differs
#   and the call can crash. Prefer fixed-argument functions.

changed through pointer: 99
p[0]: 99
type name: LP_c_int
frexp(8.0) mantissa/exponent: 0.5 4


## 06. Setting argtypes and restype

- By default ctypes assumes every argument is an int and the result is an
  int. For doubles that is WRONG and gives garbage (or crashes).
- Always set fn.argtypes (list of types) and fn.restype (return type).
- With argtypes set, Python values are converted for you and wrong counts or
  types raise ctypes.ArgumentError instead of corrupting memory.

In [6]:
if libm:
    try:
        sqrt = libm.sqrt
        sqrt.argtypes = [ctypes.c_double]
        sqrt.restype = ctypes.c_double
        print("sqrt(16.0):", sqrt(16.0))
        pow_ = libm.pow
        pow_.argtypes = [ctypes.c_double, ctypes.c_double]
        pow_.restype = ctypes.c_double
        print("pow(2, 10):", pow_(2, 10))
        try:
            sqrt("not a number")
        except ctypes.ArgumentError:
            print("wrong argument type raised ArgumentError")
    except AttributeError:
        print("libm has no sqrt on this platform")
else:
    print("libm not available; skipped")
if libc:
    libc.strlen.argtypes = [ctypes.c_char_p]
    libc.strlen.restype = ctypes.c_size_t
    print("strlen(b'hello'):", libc.strlen(b"hello"))
    libc.abs.argtypes = [ctypes.c_int]
    libc.abs.restype = ctypes.c_int
    print("abs(-7):", libc.abs(-7))

sqrt(16.0): 4.0
pow(2, 10): 1024.0
wrong argument type raised ArgumentError
strlen(b'hello'): 5
abs(-7): 7


## 07. Arrays

- Array type = element type * length, e.g. (c_int * 5).
- Behaves like a fixed-size list: index, slice, len, iterate. It cannot grow.
- Out-of-range index raises IndexError (safe), but a raw pointer would not.

In [7]:
IntArray5 = ctypes.c_int * 5
arr = IntArray5(5, 3, 8, 1, 9)
print("array:", list(arr), "len:", len(arr), "size bytes:", ctypes.sizeof(arr))
arr[0] = 50
print("slice:", arr[1:4])
try:
    arr[5]
except IndexError:
    print("index 5 -> IndexError")
print("from_buffer shares memory with a bytearray:")
ba = bytearray(b"\x01\x00\x00\x00\x02\x00\x00\x00")
view = (ctypes.c_int * 2).from_buffer(ba)
view[0] = 7
print("  bytearray now:", bytes(ba))

array: [5, 3, 8, 1, 9] len: 5 size bytes: 20
slice: [3, 8, 1]
index 5 -> IndexError
from_buffer shares memory with a bytearray:
  bytearray now: b'\x07\x00\x00\x00\x02\x00\x00\x00'


## 08. Structure

- Subclass ctypes.Structure and list _fields_ as (name, type) pairs.
- It matches a C struct, including PADDING the C compiler would add.
- _pack_ = 1 removes padding (needed for packed network/file formats).
- Union works the same way but all fields share the same memory.

In [8]:
class Point(ctypes.Structure):
    _fields_ = [("x", ctypes.c_int), ("y", ctypes.c_int)]


class Padded(ctypes.Structure):
    _fields_ = [("a", ctypes.c_char), ("b", ctypes.c_int)]


class Packed(ctypes.Structure):
    _pack_ = 1
    _fields_ = [("a", ctypes.c_char), ("b", ctypes.c_int)]


class Number(ctypes.Union):
    _fields_ = [("as_int", ctypes.c_uint32), ("as_bytes", ctypes.c_ubyte * 4)]


pt = Point(3, 4)
print("point:", pt.x, pt.y, "size:", ctypes.sizeof(pt))
print("padded vs packed size:", ctypes.sizeof(Padded), ctypes.sizeof(Packed))
print("field offset of b (padded):", Padded.b.offset)
num = Number(as_int=1)
print("union same memory, first byte:", num.as_bytes[0] in (0, 1), "sum of bytes:", sum(num.as_bytes))
print("Structure bytes:", bytes(Point(1, 2)) == struct.pack("=ii", 1, 2))
pp = ctypes.pointer(pt)
pp.contents.x = 30
print("pointer to struct changed x:", pt.x)

point: 3 4 size: 8
padded vs packed size: 8 5
field offset of b (padded): 4
union same memory, first byte: True sum of bytes: 1
Structure bytes: True
pointer to struct changed x: 30


## 09. create_string_buffer

- Python bytes are IMMUTABLE. If a C function writes into a buffer you must
  give it a mutable one: create_string_buffer(size) (bytes) or
  create_unicode_buffer (wide chars).
- .value = text up to the first NUL, .raw = all bytes.

In [9]:
buf = ctypes.create_string_buffer(16)
print("empty buffer value/raw length:", buf.value, len(buf.raw))
buf.value = b"hello"
print("after assign:", buf.value, "raw tail zeros:", buf.raw[5:] == b"\x00" * 11)
if libc:
    libc.strcpy.argtypes = [ctypes.c_char_p, ctypes.c_char_p]
    libc.strcpy.restype = ctypes.c_char_p
    libc.strcpy(buf, b"pi-ish")
    print("C wrote into our buffer:", buf.value)
print("string_at reads memory at an address:", ctypes.string_at(buf, 2))

empty buffer value/raw length: b'' 16
after assign: b'hello' raw tail zeros: True
C wrote into our buffer: b'pi-ish'
string_at reads memory at an address: b'pi'


## 10. Callbacks - qsort With CFUNCTYPE

- Some C functions take a FUNCTION POINTER (a callback). ctypes.CFUNCTYPE(
  restype, *argtypes) builds that function type; wrap a Python function in it.
- qsort(base, count, size, compare): C calls our compare(a, b) with pointers
  to two elements; we return negative / zero / positive.
- KEEP A REFERENCE to the wrapped callback while C may call it. If Python
  garbage-collects it, C calls freed memory and crashes.

In [10]:
if libc:
    CMP = ctypes.CFUNCTYPE(ctypes.c_int, ctypes.POINTER(ctypes.c_int), ctypes.POINTER(ctypes.c_int))
    calls = []

    def compare(a, b):
        calls.append(1)
        return (a[0] > b[0]) - (a[0] < b[0])

    cmp_fn = CMP(compare)  # keep this name alive!
    data = (ctypes.c_int * 6)(5, 2, 9, 1, 7, 3)
    libc.qsort.argtypes = [ctypes.c_void_p, ctypes.c_size_t, ctypes.c_size_t, CMP]
    libc.qsort.restype = None
    libc.qsort(data, len(data), ctypes.sizeof(ctypes.c_int), cmp_fn)
    print("sorted by C qsort:", list(data))
    print("Python callback was called:", len(calls) > 0)
    # Descending order, just by changing the callback
    desc = CMP(lambda a, b: (b[0] > a[0]) - (b[0] < a[0]))
    libc.qsort(data, len(data), ctypes.sizeof(ctypes.c_int), desc)
    print("descending:", list(data))
else:
    print("libc not available; qsort demo skipped")

sorted by C qsort: [1, 2, 3, 5, 7, 9]
Python callback was called: True
descending: [9, 7, 5, 3, 2, 1]


## 11. Errors and Sizes

- ctypes.get_errno() / set_errno() read C's errno, but only if you load the
  library with use_errno=True.
- On Windows: WinDLL, get_last_error(), WinError().
- ctypes.sizeof / alignment tell you the C layout; check them against the C
  header, since sizes of long, wchar_t etc. differ across platforms.
- A missing function raises AttributeError, a missing library OSError.

In [11]:
try:
    ctypes.CDLL("definitely_not_a_library_xyz.so")
except OSError:
    print("loading a missing library raises OSError")
if libc:
    try:
        libc.this_function_does_not_exist_xyz
    except AttributeError:
        print("missing symbol raises AttributeError")
    ctypes.set_errno(0)
    print("errno after set_errno(0):", ctypes.get_errno())
print("alignment of c_double is a power of 2:", ctypes.alignment(ctypes.c_double) in (1, 2, 4, 8, 16))
print("long is platform dependent; c_longlong is 8 bytes:", ctypes.sizeof(ctypes.c_longlong) == 8)

loading a missing library raises OSError
missing symbol raises AttributeError
errno after set_errno(0): 0
alignment of c_double is a power of 2: True
long is platform dependent; c_longlong is 8 bytes: True


## 12. mmap - Anonymous Memory

- mmap maps bytes into your address space. Anonymous (fileno -1) = a block of
  zeroed memory not tied to any file; handy as a big mutable buffer, or to
  share between a parent and a forked child.
- It looks like a bytearray: index, slice, assign, len, find, read/write.
- It also has a file-like cursor: read(), write(), seek(), tell().

In [12]:
with mmap.mmap(-1, 32) as m:
    print("length:", len(m), "starts zeroed:", m[:4] == b"\x00" * 4)
    m[0:5] = b"hello"
    print("slice:", m[0:5], "single index gives int:", m[0])
    m.seek(10)
    m.write(b"WORLD")
    print("tell after write:", m.tell())
    m.seek(0)
    print("read(5):", m.read(5))
    print("closed inside with:", m.closed)
print("closed after with:", m.closed)

length: 32 starts zeroed: True
slice: b'hello' single index gives int: 104
tell after write: 15
read(5): b'hello'
closed inside with: False
closed after with: True


## 13. mmap - File-Backed

- Map a real file: writes to the mapping change the file (after flush or when
  the OS decides). Reading a huge file this way avoids loading it all.
- The file must be opened with matching mode ("r+b" for read/write) and must
  NOT be empty (mapping length 0 on an empty file raises ValueError).
- access=mmap.ACCESS_READ (read-only), ACCESS_WRITE (write-through),
  ACCESS_COPY (private copy-on-write; your changes never reach the file).

In [13]:
tmpdir = tempfile.mkdtemp(prefix="mmap_demo_")
path = os.path.join(tmpdir, "data.bin")
try:
    with open(path, "wb") as f:
        f.write(b"line one\nline two\nline three\n")
    with open(path, "r+b") as f:
        with mmap.mmap(f.fileno(), 0) as mm:  # 0 = whole file
            print("size:", len(mm), "first line:", mm.readline())
            mm[0:4] = b"LINE"
            mm.flush()
    with open(path, "rb") as f:
        print("file changed on disk:", f.read(9))
    with open(path, "rb") as f:
        with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as ro:
            try:
                ro[0:1] = b"x"
            except TypeError:
                print("read-only map rejects writes (TypeError)")
    with open(path, "r+b") as f:
        with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_COPY) as cow:
            cow[0:4] = b"COPY"
            print("private copy sees:", cow[0:4])
    with open(path, "rb") as f:
        print("file still has:", f.read(4))
    empty = os.path.join(tmpdir, "empty.bin")
    open(empty, "wb").close()
    with open(empty, "r+b") as f:
        try:
            mmap.mmap(f.fileno(), 0)
        except ValueError:
            print("mapping an empty file raises ValueError")

    # 14. mmap - Slicing and find
    # ------------------------------------
    # - Slicing returns a bytes COPY of that range; m[i] returns an int.
    # - find(sub, start, end) / rfind return the index or -1 (like bytes.find).
    # - Only the pages you touch are read from disk, so find() on a giant file
    #   is memory friendly. The mmap can also be wrapped by re.search (bytes
    #   patterns work on mmap objects).
    with open(path, "r+b") as f, mmap.mmap(f.fileno(), 0) as mm:
        print("find b'two':", mm.find(b"two"), "rfind b'line':", mm.rfind(b"line"))
        print("find missing:", mm.find(b"zzz"))
        print("slice with step:", mm[0:8:2])
        print("startswith via slice:", mm[:4] == b"LINE")
        import re
        hit = re.search(rb"l\w+ t\w+", mm)
        print("regex on mmap:", hit.group())

    # 15. mmap - resize and flush
    # ------------------------------------
    # - flush() asks the OS to write changes to disk (msync). Without it, data
    #   still reaches the file eventually, but a crash may lose it.
    # - resize(n) changes the mapping AND the file length. Not supported for
    #   every access mode/platform (macOS can raise SystemError / OSError).
    # - Writes beyond the end are an error, not an extension.
    with open(path, "r+b") as f, mmap.mmap(f.fileno(), 0) as mm:
        old = len(mm)
        try:
            mm.resize(old + 10)
            print("grew by:", len(mm) - old)
            mm[old:old + 3] = b"END"
            mm.flush()
        except (SystemError, OSError, ValueError):
            print("resize not supported here; skipping")
        try:
            mm[len(mm):len(mm) + 1] = b"x"
        except (IndexError, ValueError):
            print("write past the end raises an error")
    print("file size after resize on a supporting OS is >= original:", os.path.getsize(path) >= 29)

    # 16. mmap.madvise (guarded)
    # ------------------------------------
    # - m.madvise(option) hints the OS about how you will use the pages
    #   (MADV_SEQUENTIAL, MADV_WILLNEED, MADV_DONTNEED, MADV_RANDOM ...).
    # - Python 3.8+, Unix only, and the constants vary. It is a hint: it never
    #   changes the data you read, only performance.
    with open(path, "rb") as f, mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mm:
        if hasattr(mm, "madvise") and hasattr(mmap, "MADV_SEQUENTIAL"):
            mm.madvise(mmap.MADV_SEQUENTIAL)
            print("madvise(MADV_SEQUENTIAL) accepted; data unchanged:", mm[:4])
        else:
            print("madvise not available here")
        print("PAGESIZE is a power of two:", mmap.PAGESIZE & (mmap.PAGESIZE - 1) == 0)
        print("ALLOCATIONGRANULARITY >= PAGESIZE:", mmap.ALLOCATIONGRANULARITY >= mmap.PAGESIZE)

    # 17. struct recap for binary layouts
    # ------------------------------------
    # - struct.pack/unpack convert between Python values and bytes by a format
    #   string: "<" little-endian, ">" big-endian, "=" native without padding,
    #   "@" native WITH padding (default!).
    # - Codes: b B h H i I q Q f d s (bytes) ? (bool) x (pad byte).
    # - struct.calcsize(fmt) = bytes needed; pack_into / unpack_from work
    #   directly on a buffer or mmap at an offset - no slicing copy.
    # - Use a record layout like a tiny binary file: magic, version, count.
    fmt = "<4sHI"
    header = struct.pack(fmt, b"DEMO", 1, 3)
    print("header size:", struct.calcsize(fmt), "bytes:", header)
    print("unpacked:", struct.unpack(fmt, header))
    print("native padding differs from '=':", struct.calcsize("@bi") != struct.calcsize("=bi"))
    with open(path, "r+b") as f, mmap.mmap(f.fileno(), 0) as mm:
        struct.pack_into("<I", mm, 0, 0xDEADBEEF)
        print("unpack_from mmap:", hex(struct.unpack_from("<I", mm, 0)[0]))
        rec = struct.Struct("<HH")
        rec.pack_into(mm, 4, 7, 9)
        print("Struct object reuse:", rec.unpack_from(mm, 4), "size", rec.size)
finally:
    for name in os.listdir(tmpdir):
        os.remove(os.path.join(tmpdir, name))
    os.rmdir(tmpdir)
print("temp directory cleaned up:", not os.path.exists(tmpdir))

size: 29 first line: b'line one\n'
file changed on disk: b'LINE one\n'
read-only map rejects writes (TypeError)
private copy sees: b'COPY'
file still has: b'LINE'
mapping an empty file raises ValueError
find b'two': 14 rfind b'line': 18
find missing: -1
slice with step: b'LN n'
startswith via slice: True
regex on mmap: b'line two'
resize not supported here; skipping
write past the end raises an error
file size after resize on a supporting OS is >= original: True
madvise(MADV_SEQUENTIAL) accepted; data unchanged: b'LINE'
PAGESIZE is a power of two: True
ALLOCATIONGRANULARITY >= PAGESIZE: True
header size: 10 bytes: b'DEMO\x01\x00\x03\x00\x00\x00'
unpacked: (b'DEMO', 1, 3)
native padding differs from '=': True
unpack_from mmap: 0xdeadbeef
Struct object reuse: (7, 9) size 4
temp directory cleaned up: True


## 18. Buffer Protocol and memoryview

- The BUFFER PROTOCOL is how objects share raw memory without copying.
  bytes, bytearray, array.array, mmap, ctypes arrays and numpy arrays all
  support it.
- memoryview(obj) is a window onto that memory: slicing it does NOT copy.
- See ../02-Data Types/10-Bytes-And-Bytearray.ipynb for memoryview basics.
- Here: one block of memory seen through bytearray, ctypes and mmap at once.

In [14]:
shared = bytearray(8)
mv = memoryview(shared)
as_ints = (ctypes.c_uint8 * 8).from_buffer(shared)
as_ints[0] = 65
mv[1] = 66
print("all views see the same bytes:", bytes(shared[:2]), bytes(mv[:2]), as_ints[0], as_ints[1])
sub = mv[2:6]
sub[:] = b"WXYZ"
print("memoryview slice wrote through:", bytes(shared[2:6]))
print("memoryview slice is zero-copy (same underlying obj):", sub.obj is shared)
print("format/itemsize of a memoryview:", mv.format, mv.itemsize)
as_words = mv.cast("H")
print("cast to 2-byte items, count:", len(as_words))
as_words.release()
sub.release()
mv.release()
del as_ints  # drop the ctypes view so shared can be resized again
shared.extend(b"!")
print("bytearray resizable again after views released:", len(shared))

all views see the same bytes: b'AB' b'AB' 65 66
memoryview slice wrote through: b'WXYZ'
memoryview slice is zero-copy (same underlying obj): True
format/itemsize of a memoryview: B 1
cast to 2-byte items, count: 4
bytearray resizable again after views released: 9


## 19. ctypes and mmap together

- Because both speak the buffer protocol, you can lay a Structure on top of
  an mmap with from_buffer and edit fields as attributes - the bytes change
  in the mapped memory directly. This is how you read binary file headers.

In [15]:
class Header(ctypes.Structure):
    _pack_ = 1
    _fields_ = [("magic", ctypes.c_char * 4), ("version", ctypes.c_uint16), ("count", ctypes.c_uint32)]


with mmap.mmap(-1, ctypes.sizeof(Header)) as m:
    hdr = Header.from_buffer(m)
    hdr.magic = b"DEMO"
    hdr.version = 2
    hdr.count = 1000
    print("struct size:", ctypes.sizeof(Header), "raw bytes match struct.pack:", bytes(m[:]) == struct.pack("<4sHI", b"DEMO", 2, 1000) if sys.byteorder == "little" else "skip")
    print("read back via struct:", struct.unpack_from("<4sHI", m, 0) if sys.byteorder == "little" else "skip")
    del hdr  # IMPORTANT: release the exported buffer or close() raises BufferError
print("mmap closed cleanly:", m.closed)
with mmap.mmap(-1, 8) as m2:
    held = (ctypes.c_char * 8).from_buffer(m2)
    try:
        m2.close()
    except BufferError:
        print("close() with a live ctypes view raises BufferError")
    del held

struct size: 10 raw bytes match struct.pack: True
read back via struct: (b'DEMO', 2, 1000)
mmap closed cleanly: True
close() with a live ctypes view raises BufferError


## 20. Safety Gotchas

- ctypes can SEGFAULT Python with no traceback: wrong argtypes, wrong struct
  layout, NULL pointers, buffer overruns, bad callbacks. Test in a subprocess
  or a throwaway script before trusting it.
- LIFETIME: a pointer or byref to a Python-owned buffer is only valid while
  that buffer lives. Do not return pointers to local ctypes objects.
- c_char_p given Python bytes points INTO that bytes object; never write
  through it. Use create_string_buffer for output.
- Callbacks must be kept referenced; exceptions inside them are printed and
  ignored, not raised to your caller (the C code gets a default value).
- Default restype is int (32-bit): a pointer returned from C gets truncated
  on 64-bit systems unless you set restype = c_void_p.
- mmap: slices are copies; an mmap used after close() raises ValueError;
  two processes writing the same map need their own locking.
- File size changes by another process under a mapping can crash (SIGBUS).
- A mapping offset must be a multiple of mmap.ALLOCATIONGRANULARITY.
- 32-bit vs 64-bit libs must match the Python build or loading fails.

In [16]:
if libc:
    print("an untouched function's default restype is c_int:", ctypes.CDLL(None).labs.restype is ctypes.c_int if os.name == "posix" else "skip")
m3 = mmap.mmap(-1, 8)
m3.close()
try:
    m3[0]
except ValueError:
    print("use after close raises ValueError")
try:
    mmap.mmap(-1, 8, offset=1)
except (ValueError, OSError):
    print("unaligned offset rejected")
print("callback exceptions never propagate to the C caller; plan to log inside them")

an untouched function's default restype is c_int: True
use after close raises ValueError
unaligned offset rejected
callback exceptions never propagate to the C caller; plan to log inside them


## 21. Decision Guide

- Need one or two C functions from an installed library?    -> ctypes
- Large C API, want header parsing?                         -> cffi
- Hot numeric loop, mostly Python-like code?                -> Cython / numba
- Maximum control, ship your own compiled module?           -> C extension
- Read/patch a big binary file or share memory?             -> mmap + struct
- Just parse a binary format once?                          -> read() + struct
- Pure Python solution fast enough?                         -> stay in Python

In [17]:
print("rule: reach for pure Python first, ctypes for glue, mmap for big or shared files")

rule: reach for pure Python first, ctypes for glue, mmap for big or shared files
